# 개요 

1. 문서의 내용을 읽는다
    - 참고: 소득세법 법령 https://www.law.go.kr/%EB%B2%95%EB%A0%B9/%EC%86%8C%EB%93%9D%EC%84%B8%EB%B2%95
2. 문서를 쪼갠다
    - 토큰수 초과로 답변을 생성하지 못할 수 있고 
    - 문서가 길면 (인풋이 길면) 답변 생성이 오래 걸림
3. 임베딩 -> 벡터 데이터베이스에 저장
4. 질문이 있을 때, 벡터 데이터베이스에 유사도 검색
5. 유사도 검색으로 가져온 문서를 LLM 에 질문과 같이 전달 

# 1. 문서의 내용을 읽는다

In [1]:
%pip install --upgrade --quiet docx2txt langchain-community

Note: you may need to restart the kernel to use updated packages.


In [7]:
from langchain_community.document_loaders import Docx2txtLoader

loader = Docx2txtLoader('./tax.docx')
document = loader.load()

In [ ]:
len(document) # = 1, 통으로 불러온 것 

1

# 2. 문서를 쪼갠다

In [11]:
# https://docs.langchain.com/oss/python/integrations/splitters
# https://docs.langchain.com/oss/python/integrations/splitters/recursive_text_splitter
  # default: ["\n\n", "\n", " ", ""]

%pip install -qU langchain-text-splitters

Note: you may need to restart the kernel to use updated packages.


In [6]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500, # 하나의 청크가 가지는 토큰 수
    chunk_overlap=200, # 청크 간의 겹치는 토큰 수
)


loader = Docx2txtLoader('./tax.docx')
document_list = loader.load_and_split(text_splitter=text_splitter)

In [3]:
len(document_list)

193

# 3. 임베딩 -> 벡터 데이터베이스에 저장

In [4]:
# https://developers.openai.com/api/docs/guides/embeddings

from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

In [5]:
# https://docs.langchain.com/oss/python/integrations/vectorstores/chroma
# in-memory vector store

%pip install -qU "langchain-chroma>=0.1.2"

Note: you may need to restart the kernel to use updated packages.


In [19]:
from langchain_chroma import Chroma

# document_list 를 임베딩을 이용해 디비에 저장
database = Chroma.from_documents(
    documents=document_list, 
    embedding=embeddings,
    collection_name='chroma_tax',
    persist_directory='./db/chroma'
)

# 4. 벡터 데이터베이스에 유사도 검색

In [20]:
query = "연봉 5천만원 직장인의 소득세는 얼마인가요?"
retrieved_docs = database.similarity_search(query, k=3)

In [21]:
retrieved_docs

[Document(id='18682284-4b89-456a-acd9-7c2888691a44', metadata={'source': './tax.docx'}, page_content='[전문개정 2009. 12. 31.]\n\n\n\n제10조(납세지의 변경신고) 거주자나 비거주자는 제6조부터 제9조까지의 규정에 따른 납세지가 변경된 경우 변경된 날부터 15일 이내에 대통령령으로 정하는 바에 따라 그 변경 후의 납세지 관할 세무서장에게 신고하여야 한다.\n\n[전문개정 2009. 12. 31.]\n\n\n\n제11조(과세 관할) 소득세는 제6조부터 제10조까지의 규정에 따른 납세지를 관할하는 세무서장 또는 지방국세청장이 과세한다.\n\n[전문개정 2009. 12. 31.]\n\n\n\n제2장 거주자의 종합소득 및 퇴직소득에 대한 납세의무 <개정 2009. 12. 31.>\n\n\n\n제1절 비과세 <개정 2009. 12. 31.>\n\n\n\n제12조(비과세소득) 다음 각 호의 소득에 대해서는 소득세를 과세하지 아니한다. <개정 2010. 12. 27., 2011. 7. 25., 2011. 9. 15., 2012. 2. 1., 2013. 1. 1., 2013. 3. 22., 2014. 1. 1., 2014. 3. 18., 2014. 12. 23., 2015. 12. 15., 2016. 12. 20., 2018. 3. 20., 2018. 12. 31., 2019. 12. 10., 2019. 12. 31., 2020. 6. 9., 2020. 12. 29., 2022. 8. 12., 2022. 12. 31., 2023. 8. 8., 2023. 12. 31., 2024. 12. 31., 2025. 10. 1., 2025. 12. 23.>\n\n1. 「공익신탁법」에 따른 공익신탁의 이익\n\n2. 사업소득 중 다음 각 목의 어느 하나에 해당하는 소득\n\n가. 논ㆍ밭을 작물 생산에 이용하게 함으로써 발생하는 소득\n\n나. 1개의 주택을 소유하는 자의 주택임대소득(제99조

# 5. 유사도 검색으로 가져온 문서를 LLM 에 질문과 같이 전달 

In [22]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model='gpt-4o')

In [23]:
prompt = f"""[Identity]
당신은 소득세법 전문가입니다. 
- [Context]을 참고해서 사용자의 질문에 답변해주세요.

[Context]
{retrieved_docs}

[Question]: {query}
"""

In [24]:
ai_message = llm.invoke(prompt)

In [25]:
ai_message.content

'연봉 5천만 원인 직장인의 소득세 계산은 복잡하며, 몇 가지 요소를 고려해야 합니다. 여기에서는 간단한 예시로 기본적인 계산 절차를 설명하겠습니다. 실제 소득세는 근로소득공제, 기본공제, 추가공제, 세액공제 등 여러 요소를 고려하여 산출됩니다.\n\n1. **근로소득공제 계산**:\n   - 근로소득공제는 총급여액에 따라 다르게 적용되며, 일정 비율과 금액으로 구성됩니다. 예를 들어, 총급여액이 5천만 원일 때는 대략 900만원 정도의 근로소득공제가 적용됩니다.\n\n2. **과세표준 계산**:\n   - 과세표준은 연봉에서 근로소득공제, 기본공제 (가족 수에 따라 다름) 및 기타공제를 차감하여 계산합니다.\n   - 가정하여 기본공제(1인 기준)를 적용했다고 가정하면, 과세표준은 대략 5천만 원 - 900만 원 - 기본공제(150만 원) ≈ 3850만 원입니다.\n\n3. **소득세 세율 적용**:\n   - 과세표준에 따라 누진세율이 적용됩니다. 과세표준이 3850만 원인 경우, 대략적으로 15%의 세율이 적용될 수 있습니다.\n   - 따라서, 소득세는 약 577.5만 원 (3850만 원 x 15%)입니다.\n\n4. **세액공제 및 연말정산**:\n   - 최종적으로 소득세에서 세액공제 및 연말정산을 통해 실제 납부할 세액이 결정됩니다.\n\n위의 계산은 기본적인 예시에 불과하며, 각 개인의 상황에 따라 달라질 수 있습니다. 또한, 세법은 자주 개정되므로, 최신 정보를 바탕으로 세무 전문가에게 문의하는 것이 가장 정확합니다.'